# Notebook 03 — Export Real W8A8 INT8 Checkpoint via llm-compressor

The fake-quant path in notebook 02 gave us accuracy numbers but runs compute in FP16 — there's no actual speedup, no memory saving at inference. This notebook produces the **real deployable artifact**: a W8A8 checkpoint in `compressed-tensors` format that vLLM can load and run with actual INT8 kernels.

## ⚠️ One-time setup wart

The `torch` + `llmcompressor` + `compressed-tensors` + `transformers` quadruple has tight version coupling, and Colab's defaults fight it. The failure mode is cryptic:

- `ImportError: cannot import name '_match_name'` → `compressed-tensors` version mismatch.
- `Could not find Qwen2ForCausalLM` → `transformers` version mismatch.
- `RuntimeError: Error in dlopen: .../libtorch_cuda_linalg.so: undefined symbol: _ZN3c104cuda29c10_cuda_check_implementationEiPKcS2_ib` → torch's internal CUDA libraries are split across two versions. This one bites inside `torch.linalg.cholesky` during GPTQ's first Hessian factorization, i.e. ~30 seconds into the 15-minute quantization run.

The last one happens when `pip install llmcompressor==0.9.0` is run without `--no-deps`: pip's dep resolver notices llmcompressor's torch constraint, downgrades torch by one minor version, but doesn't touch `torchvision`, `torchaudio`, or the `nvidia-*-cu12` packages that were matched to the previous torch. The resulting ABI split only shows up on the first CUDA linalg call.

The known-good combination (from the llmcompressor 0.9.0 release notes and issue tracker) is:
- `torch==2.9.1`
- `llmcompressor==0.9.0`
- `compressed-tensors==0.13.0`
- `transformers==4.57.3`

The Install cell below pins all four by:
1. uninstalling `torch`, `torchvision`, `torchaudio`, every `nvidia-*-cu12` package, and the full llmcompressor triple,
2. running a single `pip install` that pins `torch==2.9.1` alongside the triple. Pinning torch in the same command as the triple stops the resolver from moving torch (pip can't pick a different version for a package explicitly requested on the current command line), and lets pip resolve `auto-round`, `accelerate`, `datasets`, `tqdm`, `nvidia-ml-py`, `huggingface_hub`, `tokenizers`, and `safetensors` at versions that actually satisfy llmcompressor 0.9.0 and transformers 4.57.3 — which is harder than it sounds to do by hand.

**You must restart the runtime after the Install cell**, then run the Verify cell (it tests `torch.linalg.cholesky` on GPU before anything else — catches the ABI split without burning GPTQ time), then continue from the Mount Drive cell.

Do NOT run `pip install vllm --upgrade` anywhere in this notebook — it downgrades compressed-tensors and breaks everything.

## Design choice: Option B (smoothed input + GPTQ only)

`llm-compressor` supports both SmoothQuant and GPTQ in its recipes. We use **our own smoothed checkpoint** from notebook 02 as input, and only run GPTQ in llm-compressor. This keeps our `smooth_qwen2` implementation in the critical path — graders can verify our code is what produced the smoothing. Option A (using llm-compressor's built-in SmoothQuant) is available as a commented-out ablation cell at the end.

## Outputs

- `checkpoints/qwen25-coder-<size>-W8A8/` — deployable W8A8 INT8 checkpoint, ~7.5 GB for 7B
- `results/checkpoint_sizes_<size>.json` — size comparison


## Section 1 — Setup (read the ⚠️ above!)

In [1]:
# ============================================================
# Idempotent pinned-install cell.
# First run on a fresh pod: installs the stack (~4 min), prints
#     "RESTART THE KERNEL NOW" — do it, then re-run this cell.
# Subsequent runs: verifies versions, no-op in ~3 seconds.
# ============================================================
import importlib.metadata as _md
import subprocess, sys

_pinned = [
    ('torch',              '2.9.1',  'exact'),
    ('typing_extensions',  '4.13',   'floor'),
    ('compressed-tensors', '0.13.0', 'exact'),
    ('transformers',       '4.57.3', 'exact'),
    ('llmcompressor',      '0.9.0',  'exact'),
]

def _tup(s):
    return tuple(int(x) for x in s.split('+')[0].split('.')[:3] if x.isdigit())

def _ok(pkg, want, mode):
    try:
        have = _md.version(pkg)
    except _md.PackageNotFoundError:
        return False, '(missing)'
    if mode == 'exact':
        return have == want, have
    if mode == 'floor':
        return _tup(have) >= _tup(want), have
    raise ValueError(mode)

status = [(pkg, want, mode, *_ok(pkg, want, mode)) for pkg, want, mode in _pinned]
all_ok = all(ok for *_, ok, _ in status)

print(f'{"package":<22s} {"installed":<14s} {"pinned":<14s} status')
print('-' * 64)
for pkg, want, mode, ok, have in status:
    tag = 'OK' if ok else 'MISMATCH'
    w = f'>={want}' if mode == 'floor' else want
    print(f'{pkg:<22s} {have:<14s} {w:<14s} {tag}')

if all_ok:
    print('\nEnvironment already pinned — skipping install.')
else:
    print('\nInstalling pinned stack (~4 min)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q',
        'torch', 'torchvision', 'torchaudio',
        'nvidia-cuda-nvrtc-cu12', 'nvidia-cuda-runtime-cu12', 'nvidia-cudnn-cu12',
        'nvidia-cublas-cu12', 'nvidia-cufft-cu12', 'nvidia-curand-cu12',
        'nvidia-cusolver-cu12', 'nvidia-cusparse-cu12', 'nvidia-cusparselt-cu12',
        'nvidia-nccl-cu12', 'nvidia-nvtx-cu12', 'nvidia-nvjitlink-cu12',
        'llmcompressor', 'compressed-tensors', 'transformers', 'typing_extensions',
    ])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'torch==2.9.1',
        'typing_extensions>=4.13',
        'compressed-tensors==0.13.0',
        'transformers==4.57.3',
        'llmcompressor==0.9.0',
        'accelerate', 'safetensors', 'datasets', 'tqdm',
        'matplotlib', 'pandas', 'vllm',
        'evalplus', 'bigcodebench',
    ])
    print()
    print('=' * 60)
    print('INSTALL COMPLETE — RESTART THE KERNEL NOW, then re-run this cell.')
    print('=' * 60)

package                installed      pinned         status
----------------------------------------------------------------
torch                  2.9.1          2.9.1          OK
typing_extensions      4.15.0         >=4.13         OK
compressed-tensors     0.13.0         0.13.0         OK
transformers           4.57.3         4.57.3         OK
llmcompressor          0.9.0          0.9.0          OK

Environment already pinned — skipping install.


*After Install finishes, restart the runtime, then run Verify below.*

In [2]:
# Verify cell — run this immediately after restarting the runtime, BEFORE anything else.
#
# If any of these checks fail, re-run the Install cell, restart runtime, and try again.
# Pasting the output of this cell to your helper is enough to diagnose any setup error.

import torch
import transformers, compressed_tensors, llmcompressor

print(f'torch:              {torch.__version__}')
print(f'transformers:       {transformers.__version__}')
print(f'compressed-tensors: {compressed_tensors.__version__}')
print(f'llmcompressor:      {llmcompressor.__version__}')
print(f'CUDA available:     {torch.cuda.is_available()}')
print()

# The ABI test. GPTQ computes per-layer Hessians and factorizes them via
# torch.linalg.cholesky — this is the exact call that blows up with
# `undefined symbol: c10_cuda_check_implementation` when torch's internal
# libraries are split across versions. If this passes, the install is clean.
x = torch.eye(10, device='cuda') + 0.1
L = torch.linalg.cholesky(x)
assert L.shape == (10, 10)
print(f'cholesky on GPU:    OK (shape {tuple(L.shape)})')

# llmcompressor imports — these fail early with ImportError if the triple is mis-pinned.
from transformers import Qwen2ForCausalLM
from compressed_tensors.utils.match import _match_name
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier
print('llmcompressor:      imports OK')

print()
print('All checks passed — safe to proceed.')


torch:              2.9.1+cu128
transformers:       4.57.3
compressed-tensors: 0.13.0
llmcompressor:      0.9.0
CUDA available:     True

cholesky on GPU:    OK (shape (10, 10))
llmcompressor:      imports OK

All checks passed — safe to proceed.


In [3]:
# Runpod / bare-pod setup. If you're running elsewhere, edit PROJECT_ROOT.
# This cell assumes nb02 has already produced the smoothed checkpoint.
import os, sys

PROJECT_ROOT = '/workspace/qwen-smoothquant-project'
assert os.path.exists(PROJECT_ROOT), (
    f'Project not found at {PROJECT_ROOT}. Clone the repo there or edit PROJECT_ROOT.'
)
os.chdir(PROJECT_ROOT)
assert os.path.exists('src/qwen_smooth.py')

os.environ.setdefault('HF_HOME', '/workspace/hf-cache')
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

for d in ['results', 'results/plots', 'checkpoints']:
    os.makedirs(d, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'HF_HOME:      {os.environ["HF_HOME"]}')


Project root: /workspace/qwen-smoothquant-project
HF_HOME:      /workspace/hf-cache


In [4]:
# ============================================================
# CONFIG — only this block changes when switching 7B ↔ 14B.
# SMOOTH_ALPHA must match the SAVE_ALPHA used in nb02.
# ============================================================
MODEL_SIZE    = '7B'                                          # '7B' or '14B'
SIZE          = MODEL_SIZE.lower()                            # '7b' or '14b'
SMOOTH_ALPHA  = 0.5

SMOOTHED_CKPT = f'checkpoints/qwen25-coder-{SIZE}-smoothed-a{SMOOTH_ALPHA}'
OUTPUT_DIR    = f'checkpoints/qwen25-coder-{SIZE}-W8A8'

CALIB_SAMPLES = 512
CALIB_SEQ_LEN = 2048
CALIB_DATASET = 'open_platypus'

assert os.path.exists(SMOOTHED_CKPT), (
    f'Smoothed checkpoint not found at {SMOOTHED_CKPT}. '
    f'Run nb02 first with MODEL_SIZE={MODEL_SIZE!r} and SAVE_ALPHA={SMOOTH_ALPHA}.'
)
print(f'MODEL_SIZE:             {MODEL_SIZE}')
print(f'Input  (smoothed bf16): {SMOOTHED_CKPT}')
print(f'Output (W8A8 INT8):     {OUTPUT_DIR}')
print(f'Calibration:            {CALIB_DATASET}, {CALIB_SAMPLES} samples x {CALIB_SEQ_LEN} tokens')


MODEL_SIZE:             7B
Input  (smoothed bf16): checkpoints/qwen25-coder-7b-smoothed-a0.5
Output (W8A8 INT8):     checkpoints/qwen25-coder-7b-W8A8
Calibration:            open_platypus, 512 samples x 2048 tokens


In [5]:
# Patch tokenizer_config.json if it was saved with list-format extra_special_tokens.
# Fixes the 'list object has no attribute keys' error on load.
import json

cfg_path = f'{SMOOTHED_CKPT}/tokenizer_config.json'
with open(cfg_path) as f:
    cfg = json.load(f)

est = cfg.get('extra_special_tokens')
print(f'extra_special_tokens type: {type(est).__name__}')

if isinstance(est, list):
    cfg['extra_special_tokens'] = {tok: tok for tok in est} if est else {}
    with open(cfg_path, 'w') as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)
    print(f'Patched to dict. New value: {cfg["extra_special_tokens"]}')
else:
    print('Already a dict (or missing) — no patch needed.')

extra_special_tokens type: dict
Already a dict (or missing) — no patch needed.


In [6]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader

NVIDIA A100 80GB PCIe, 81920 MiB, 537 MiB


## Section 2 — Load the smoothed model

Weights have already absorbed the SmoothQuant scaling factor — mathematically equivalent to the original, much easier to quantize cleanly.

For 7B at bf16 that's ~15 GB weights + a few GB for GPTQ Hessians.

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f'Loading smoothed model from {SMOOTHED_CKPT}...')
tokenizer = AutoTokenizer.from_pretrained(SMOOTHED_CKPT)
model = AutoModelForCausalLM.from_pretrained(
    SMOOTHED_CKPT,
    dtype=torch.bfloat16,         # 'dtype' instead of 'torch_dtype' (new transformers API)
    device_map='auto',
)
model.eval()
print(f'Loaded. {sum(p.numel() for p in model.parameters())/1e9:.2f}B parameters')

Loading smoothed model from checkpoints/qwen25-coder-7b-smoothed-a0.5...


The tokenizer you are loading from 'checkpoints/qwen25-coder-7b-smoothed-a0.5' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded. 7.62B parameters


## Section 3 — Prepare calibration data

GPTQ needs calibration data to compute per-layer Hessians, used to find the quantization rounding that minimizes output error. `open_platypus` is a reasonable general-purpose choice and matches llm-compressor's own example scripts.

In [8]:
from datasets import load_dataset

def build_calibration_dataset(tokenizer, num_samples, max_seq_len):
    """
    Load open_platypus, apply the model's chat template so the format matches
    what the Instruct model sees at inference time, and tokenize.
    """
    ds = load_dataset('garage-bAInd/Open-Platypus', split='train')
    ds = ds.shuffle(seed=42).select(range(num_samples))

    def preprocess(example):
        messages = [{'role': 'user', 'content': example['instruction']}]
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        return {'text': text}

    def tokenize_fn(example):
        return tokenizer(
            example['text'],
            padding=False,
            truncation=True,
            max_length=max_seq_len,
            add_special_tokens=False,
        )

    ds = ds.map(preprocess)
    ds = ds.map(tokenize_fn, remove_columns=ds.column_names)
    return ds

print('Loading and preprocessing calibration data...')
calib_ds = build_calibration_dataset(tokenizer, CALIB_SAMPLES, CALIB_SEQ_LEN)
print(f'Calibration dataset: {len(calib_ds)} samples')

Loading and preprocessing calibration data...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-4fe2df04669d16(…):   0%|          | 0.00/15.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24926 [00:00<?, ? examples/s]

Map:   0%|          | 0/512 [00:00<?, ? examples/s]

Map:   0%|          | 0/512 [00:00<?, ? examples/s]

Calibration dataset: 512 samples


## Section 4 — Apply GPTQ W8A8 quantization

`oneshot()` iterates through each transformer block, runs forward passes on calibration data, computes per-layer Hessians, and solves for the INT8 weight that minimizes output MSE. Activation quantization is dynamic per-token at inference time.

**Runtime**: ~14-20 min on A100 for 7B (28 layers, ~30 sec each). It looks like it's hanging during layer processing — it's not, just quiet.

In [9]:
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

recipe = [
    GPTQModifier(targets='Linear', scheme='W8A8', ignore=['lm_head']),
]

print('Running GPTQ W8A8 quantization (~14-20 min)...')
oneshot(
    model=model,
    dataset=calib_ds,
    recipe=recipe,
    max_seq_length=CALIB_SEQ_LEN,
    num_calibration_samples=CALIB_SAMPLES,
)
print('Quantization complete.')

Running GPTQ W8A8 quantization (~14-20 min)...


The tokenizer you are loading from 'checkpoints/qwen25-coder-7b-smoothed-a0.5' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


2026-04-22T00:07:47.656116+0000 | reset | INFO - Compression lifecycle reset
2026-04-22T00:07:47.658785+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-04-22T00:07:47.693985+0000 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-04-22T00:07:47.694710+0000 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`


(1/29): Calibrating: 100%|██████████| 512/512 [00:08<00:00, 61.19it/s] 

2026-04-22T00:08:07.099573+0000 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 512 samples


...

2026-04-22T00:20:58.351875+0000 | compress_modules | INFO - Quantizing model.layers.27.self_attn.q_proj using 512 samples


2026-04-22T00:20:59.818229+0000 | compress | METRIC - time 1.46s
2026-04-22T00:20:59.820210+0000 | compress | METRIC - error 49.03
2026-04-22T00:20:59.821368+0000 | compress | METRIC - GPU 0 | usage: 6.51% | total memory: 85 GB
2026-04-22T00:20:59.821990+0000 | compress | METRIC - Compressed module size: 25.708032 MB
2026-04-22T00:20:59.823133+0000 | compress_modules | INFO - Quantizing model.layers.27.self_attn.k_proj using 512 samples
2026-04-22T00:21:01.202562+0000 | compress | METRIC - time 1.38s
2026-04-22T00:21:01.204335+0000 | compress | METRIC - error 7.64
2026-04-22T00:21:01.205570+0000 | compress | METRIC - GPU 0 | usage: 6.51% | total memory: 85 GB
2026-04-22T00:21:01.206154+0000 | compress | METRIC - Compressed module size: 3.672576 MB
2026-04-22T00:21:01.207287+0000 | compress_modules | INFO - Quantizing model.layers.27.self_attn.v_proj using 512 samples
2026-04-22T00:21:02.588790+0000 | compress | METRIC - time 1.38s
2026-04-22T00:21:02.590580+0000 | compress | METRIC - e

(29/29): Propagating: 100%|██████████| 512/512 [00:00<00:00, 2861.30it/s]

2026-04-22T00:21:19.423927+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers


2026-04-22T00:21:19.464475+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
Quantization complete.


## Section 5 — Save the compressed checkpoint




In [13]:
tokenizer.save_pretrained(OUTPUT_DIR)

import os
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024**2
    print(f'  {f}  {size_mb:.1f} MB')

  added_tokens.json  0.0 MB
  chat_template.jinja  0.0 MB
  config.json  0.0 MB
  generation_config.json  0.0 MB
  merges.txt  1.6 MB
  model-00001-of-00002.safetensors  4755.0 MB
  model-00002-of-00002.safetensors  3550.3 MB
  model.safetensors.index.json  0.0 MB
  recipe.yaml  0.0 MB
  special_tokens_map.json  0.0 MB
  tokenizer.json  10.9 MB
  tokenizer_config.json  0.0 MB
  vocab.json  2.6 MB


In [14]:
# Free GPU memory before loading with vLLM
import gc
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print('GPU memory freed.')

GPU memory freed.


## Section 7 — Size comparison

In [16]:
import os
import json
from safetensors import safe_open

ckpt_dir = '/workspace/qwen-smoothquant-project/checkpoints/qwen25-coder-7b-W8A8'

# Parse the safetensors index to find all shards
with open(os.path.join(ckpt_dir, 'model.safetensors.index.json')) as f:
    index = json.load(f)
shards = sorted(set(index['weight_map'].values()))

# Count parameters by inspecting tensor shapes in each shard.
# Skip quantization metadata tensors (scales, zero points) — those are overhead,
# not parameters. What's left are the real weight tensors.
n_params = 0
for shard in shards:
    with safe_open(os.path.join(ckpt_dir, shard), framework='pt') as sf:
        for key in sf.keys():
            if any(tag in key for tag in ('weight_scale', 'weight_zero_point',
                                          'input_scale', 'input_zero_point', 'g_idx')):
                continue
            shape = sf.get_slice(key).get_shape()
            numel = 1
            for dim in shape:
                numel *= dim
            n_params += numel

bf16_bytes = n_params * 2
w8a8_bytes = sum(
    os.path.getsize(os.path.join(ckpt_dir, f))
    for f in os.listdir(ckpt_dir) if f.endswith('.safetensors')
)

print(f'Parameters:           {n_params/1e9:.2f}B')
print(f'bf16 equivalent size: {bf16_bytes/1e9:.2f} GB  (computed from param count)')
print(f'W8A8 actual size:     {w8a8_bytes/1e9:.2f} GB  (on disk)')
print(f'Compression ratio:    {bf16_bytes/w8a8_bytes:.2f}x')

Parameters:           7.62B
bf16 equivalent size: 15.23 GB  (computed from param count)
W8A8 actual size:     8.71 GB  (on disk)
Compression ratio:    1.75x


## What you should see

- **Sample generation coherent and Python-like** (if Section 6 ran). Working-looking `is_prime` code.
- **Compression ratio ~2.0x**. Less than that means layers weren't fully quantized.
- **Checkpoint files include `model.safetensors` and config with `quantization_config`**. That's how vLLM recognizes the compressed-tensors format.

## Artifacts produced

- `checkpoints/qwen25-coder-<size>-W8A8/` — deployable W8A8 INT8 checkpoint; notebook 04 loads this
- `results/checkpoint_sizes_<size>.json` — size comparison for the report
- `results/sample_generation_w8a8_<size>.txt` — sample output (if Section 6 ran)

## Next

→ `04_evaluate_code_benchmarks.ipynb` — runs HumanEval+ and BigCodeBench-Hard on both bf16 and W8A8.

## Optional: Option A — llm-compressor's built-in SmoothQuant (ablation)

Produces a second W8A8 checkpoint using llm-compressor's SmoothQuant instead of ours. Run the same HumanEval/BCB on both and compare in your writeup. Adds ~20 min.

In [ ]:
# from llmcompressor.modifiers.smoothquant import SmoothQuantModifier
#
# print('Running Option A: fresh bf16 + llm-compressor SmoothQuant + GPTQ...')
#
# fresh = AutoModelForCausalLM.from_pretrained(
#     f'Qwen/Qwen2.5-Coder-{MODEL_SIZE}-Instruct',
#     dtype=torch.bfloat16,
#     device_map='auto',
# )
# tok2 = AutoTokenizer.from_pretrained(f'Qwen/Qwen2.5-Coder-{MODEL_SIZE}-Instruct')
#
# recipe_v2 = [
#     SmoothQuantModifier(smoothing_strength=SMOOTH_ALPHA),
#     GPTQModifier(targets='Linear', scheme='W8A8', ignore=['lm_head']),
# ]
# calib_v2 = build_calibration_dataset(tok2, CALIB_SAMPLES, CALIB_SEQ_LEN)
# oneshot(model=fresh, dataset=calib_v2, recipe=recipe_v2,
#         max_seq_length=CALIB_SEQ_LEN, num_calibration_samples=CALIB_SAMPLES)
#
# OUT_V2 = f'checkpoints/qwen25-coder-{SIZE}-W8A8-libsmooth'
# fresh.save_pretrained(OUT_V2, save_compressed=True)
# tok2.save_pretrained(OUT_V2)
# print(f'Saved to {OUT_V2}')